# COVID-19 Label Classification (Educational Example)

This notebook demonstrates a basic **Logistic Regression** classification workflow using the supplied `covid.csv` file.

> **Safety and scope:** This is an educational exercise, not a medical model. The dataset's provenance, label definitions, collection method, and suitability for any real-world use were not supplied. Do not use this model for diagnosis, screening, or healthcare decisions.

## Project goals
- Inspect and validate tabular data.
- Separate input features from the target label.
- Split data reproducibly while preserving class proportions.
- Scale numeric features within a pipeline and fit Logistic Regression.
- Evaluate with accuracy, precision, recall, F1-score, and a confusion matrix.

## Dataset fields
The supplied file has the columns `Age`, `Gender`, `Fever`, `Cough`, and `Has_Covid`. The meaning of the numeric encodings should be confirmed from the dataset documentation before interpreting them.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("covid.csv")
FEATURE_COLUMNS = ["Age", "Gender", "Fever", "Cough"]
TARGET_COLUMN = "Has_Covid"


## 1. Load and inspect the data


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Put covid.csv beside this notebook."
    )

df = pd.read_csv(DATA_PATH)

display(df.head())
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing values"))


## 2. Validate columns and handle missing values

The supplied file contains missing values. This notebook drops incomplete rows for a simple baseline and reports how many rows remain. This is a modeling choice, not a universally best way to handle missing data.


In [ ]:
required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

data = df[required_columns].copy()
rows_before = len(data)
data = data.dropna()
rows_dropped = rows_before - len(data)

if data.empty:
    raise ValueError("No complete rows remain after dropping missing values.")

# The supplied target is encoded as 0/1. Validate before converting to integers.
target_values = set(data[TARGET_COLUMN].unique())
if not target_values.issubset({0, 1, 0.0, 1.0}):
    raise ValueError(f"Expected a binary 0/1 target, found: {sorted(target_values)}")

X = data[FEATURE_COLUMNS]
y = data[TARGET_COLUMN].astype(int)

if y.nunique() != 2:
    raise ValueError("Both target classes must be present to run this example.")

print(f"Rows before cleaning: {rows_before}")
print(f"Rows dropped: {rows_dropped}")
print(f"Rows used: {len(data)}")
display(y.value_counts().sort_index().rename("count").to_frame())


## 3. Split the data

`stratify=y` helps preserve the class proportions in both sets. The fixed seed makes the split reproducible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")


## 4. Train a scaled Logistic Regression pipeline

Scaling is fitted only on the training data because it is inside the scikit-learn pipeline. This prevents test-set information from leaking into preprocessing.


In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


## 5. Evaluate the model

Accuracy alone can hide poor performance on one class. Review precision, recall, F1-score, and the confusion matrix together. These are measurements on this dataset split only; they do not establish medical validity.


In [ ]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print()
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        zero_division=0,
        target_names=["Label 0", "Label 1"],
    )
)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
display(pd.DataFrame(
    cm,
    index=["Actual label 0", "Actual label 1"],
    columns=["Predicted label 0", "Predicted label 1"],
))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Label 0", "Label 1"],
).plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Held-out Test Set")
plt.tight_layout()
plt.show()


## 6. Optional: inspect a fictional example

This cell demonstrates the model's input format only. It is **not** a health assessment, and its output must not be interpreted as a diagnosis. Use only values that conform to the dataset's documented encodings and units.


In [ ]:
example = pd.DataFrame(
    [
        {
            "Age": 35,
            "Gender": 0,
            "Fever": 100.4,
            "Cough": 1,
        }
    ],
    columns=FEATURE_COLUMNS,
)

example_prediction = model.predict(example)[0]
print(f"Predicted dataset label: {example_prediction}")


## Conclusion

This project demonstrates a reproducible classification pipeline and several evaluation metrics. The supplied dataset's source and label semantics are unknown, and the observed test performance is limited. The model is for learning only.

### Before public release
- Find and cite the original dataset source and license.
- Document what labels `0` and `1` mean and the units/encodings of every feature.
- Re-run all cells from top to bottom in a clean environment.
- Do not describe this notebook as a COVID diagnosis or screening tool.
